# `quotationline` — bidder's priced BOQ line items

Unity Catalog: `ingestion_framework_test.bid_data_exploration.quotationline`

Expected: the real equivalent of the per-item CIF/Erection rows the sample data's Excel/PDF parsers currently extract by hand. Look for: item number, description, unit, qty, rate(s), total(s), whether CIF/Erection are separate columns or need to be derived, and the foreign key back to `rfqvendor`.

## Run 1 — initial exploration ✅ *(run in Databricks — awaiting results to document findings)*

In [0]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.quotationline

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM ingestion_framework_test.bid_data_exploration.quotationline

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.quotationline LIMIT 20

## Run 2 — follow-up queries ⏳ *(pending — not yet run)*

First pass findings (full write-up in `databricks/FINDINGS.md`):
- **`BOQITEMNUM` is a real, dedicated column** — almost certainly the native equivalent of the sample data's hierarchical item numbers (`1.2.3`). Null in this sample (these are catalog/material RFQs, not detailed-BOQ ones) — need to find a populated example.
- **No separate CIF/Erection columns** — just one `UNITCOST`/`LINECOST` pair per line. Leading theory: the `LINETYPE` column (`MATERIAL` vs. `SERVICE` seen in the sample) is how CIF (supply) vs. Erection (install) get split — as two separate line rows sharing the same `BOQITEMNUM`, not two columns on one row. Needs confirming.
- **`ISAWARDED` is tracked per line, per vendor** — confirmed in the sample: for tender `D19885308`, 3 vendors (002537, 003318, 9929192) each quoted the same items, and only vendor 002537's lines 4-5 show `ISAWARDED=1`. This is the real award-decision mechanism, line by line, not just a single winner per RFQ.
- **Correction to my last note in `06_cross_table_relationships`:** there is **no `RFQVENDORID`/`RFQVENDOR_ID` column in `quotationline` at all** — I was wrong to treat the earlier uncommented query as confirming that column exists (outputs were stripped, so it may well have errored rather than succeeded — I shouldn't have inferred success from it being left uncommented). The real join back to `rfqvendor` looks like the **composite key `(RFQNUM, VENDOR)`**, which both tables have.
- Sample shows RFQs with 500+ sequential `RFQLINENUM`s (`D-104880` at line 557-559) — consistent with real large BOQs having ~218 rows per lot like the sample data.
- A third org appears (`TRANS`/`TRANSORG`) beyond `ADWEA`/`ADDC` — reinforces that this dataset spans the whole group, not just ADDC.

### Find a populated `BOQITEMNUM` example (a real detailed-BOQ tender)

In [ ]:
%sql
SELECT RFQNUM, VENDOR, BOQITEMNUM, RFQLINENUM, DESCRIPTION, ORDERQTY, UNITCOST, LINECOST, LINETYPE
FROM ingestion_framework_test.bid_data_exploration.quotationline
WHERE BOQITEMNUM IS NOT NULL
ORDER BY ENTERDATE DESC
LIMIT 30

### Is `LINETYPE` the CIF/Erection split? (distribution first)

In [ ]:
%sql
SELECT LINETYPE, COUNT(*) AS n FROM ingestion_framework_test.bid_data_exploration.quotationline
GROUP BY LINETYPE ORDER BY n DESC

### Does the same `BOQITEMNUM` appear twice per vendor with different `LINETYPE`s?
If CIF/Erection are split into two rows, expect `COUNT(*) = 2` for most groups below.

In [ ]:
%sql
SELECT RFQNUM, VENDOR, BOQITEMNUM, COUNT(*) AS n, COLLECT_SET(LINETYPE) AS linetypes
FROM ingestion_framework_test.bid_data_exploration.quotationline
WHERE BOQITEMNUM IS NOT NULL
GROUP BY RFQNUM, VENDOR, BOQITEMNUM
ORDER BY n DESC
LIMIT 30

### Which RFQs actually look like real detailed BOQs (100+ line items)?

In [ ]:
%sql
SELECT RFQNUM, COUNT(*) AS line_count, COUNT(DISTINCT VENDOR) AS vendor_count
FROM ingestion_framework_test.bid_data_exploration.quotationline
GROUP BY RFQNUM
HAVING COUNT(*) > 100
ORDER BY line_count DESC
LIMIT 30

### Does D-111808 (or similar) exist directly in this table?

In [ ]:
%sql
SELECT RFQNUM, VENDOR, BOQITEMNUM, DESCRIPTION, ORDERQTY, UNITCOST, LINECOST, LINETYPE, ISAWARDED
FROM ingestion_framework_test.bid_data_exploration.quotationline
WHERE RFQNUM LIKE 'D-111808%'
ORDER BY VENDOR, RFQLINENUM
LIMIT 50

### Confirm the (RFQNUM, VENDOR) join back to `rfqvendor` actually lines up

In [ ]:
%sql
SELECT ql.RFQNUM, ql.VENDOR, COUNT(*) AS line_count, SUM(ql.LINECOST) AS total_linecost
FROM ingestion_framework_test.bid_data_exploration.quotationline ql
GROUP BY ql.RFQNUM, ql.VENDOR
ORDER BY line_count DESC
LIMIT 20

**Observations (updated after first real run):**
- `BOQITEMNUM` is the real hierarchical item number column — needs a populated example to confirm format matches `1.2.3`-style numbering.
- No CIF/Erection columns — `LINETYPE` (`MATERIAL`/`SERVICE`) splitting into two rows per `BOQITEMNUM` is the leading theory, unconfirmed.
- `ISAWARDED` genuinely tracks award status per vendor per line — this is real, granular award data, richer than anything in the sample model.
- Join back to `rfqvendor` is the composite key `(RFQNUM, VENDOR)`, **not** a single `RFQVENDORID`/`RFQVENDOR_ID` FK column — that column does not exist in this table.